# `DEFINE SEQUENCE` — Sequential IDs & Auto-Numbered Fields

SurrealDB normally assigns random string IDs (`invoice:3xk9f2...`). SurrealEngine 0.9.8 adds two complementary sequence features:

| Mechanism | What it does |
|---|---|
| `Meta.sequence` | Auto-assigns clean integer **record IDs** (`invoice:1`, `invoice:2`, …) |
| `SequenceField` | Auto-numbers a regular **field** (e.g. `order_number=1000`); record ID stays random |

In [ ]:
from surrealengine import (
    Document, StringField, IntField, FloatField,
    SequenceField,
    create_connection,
)

## 1. Integer record IDs via `Meta.sequence`

Set `sequence` (and optionally `sequence_start` / `sequence_batch`) in `Meta`.
`create_table()` emits `DEFINE SEQUENCE`, and every `save()` automatically fetches
the next value and uses it as the record ID.

In [ ]:
class Invoice(Document):
    """Invoice with a clean sequential integer ID."""
    customer = StringField(required=True)
    amount   = FloatField(required=True)
    status   = StringField(default="pending")

    class Meta:
        collection     = "invoice"
        sequence       = "invoice_seq"  # enables auto-ID from this sequence
        sequence_start = 1              # start at 1
        sequence_batch = 1              # increment by 1

## 2. Connect and register the schema

In [ ]:
conn = create_connection(
    url       = "ws://db:8000/rpc",
    namespace = "test_ns",
    database  = "test_db",
    username  = "root",
    password  = "secret",
    make_default = True,
)
await conn.connect()
print("Connected")

# Emits both DEFINE TABLE and DEFINE SEQUENCE IF NOT EXISTS invoice_seq BATCH 1 START 1
await Invoice.create_table()
print("Invoice table + sequence ready")

## 3. Create invoices — no ID management needed

`save()` calls `sequence::nextval('invoice_seq')` automatically and assigns
the result as an integer record ID.

In [ ]:
for customer, amount in [
    ("Acme Corp", 1500.00),
    ("Globex",     750.50),
    ("Initech",   3200.00),
]:
    inv = Invoice(customer=customer, amount=amount)
    await inv.save()  # ID is assigned automatically from invoice_seq
    print(f"{inv.id}  customer={inv.customer}  amount={inv.amount}")

## 4. Verify the IDs are clean integers

In [ ]:
all_invoices = await Invoice.objects.order_by("id").all()
for inv in all_invoices:
    print(f"{inv.id}  →  {inv.customer}  ${inv.amount:.2f}  [{inv.status}]")
# Expected: invoice:1, invoice:2, invoice:3  (no backticks, no random strings)

## 5. Auto-numbered fields via `SequenceField`

Use `SequenceField` when you want a sequential number on a **field**, not the record ID.
`create_table()` emits the `DEFINE SEQUENCE` DDL; `save()` fetches `nextval()` automatically.

In [ ]:
class Order(Document):
    # order_number is filled from a sequence; record ID stays random
    order_number = SequenceField(sequence="order_seq", start=1000, batch=10)
    customer     = StringField()
    total        = FloatField()

    class Meta:
        collection = "order"
        # No sequence here — record ID will be random


# Emits DEFINE TABLE order + DEFINE SEQUENCE IF NOT EXISTS order_seq BATCH 10 START 1000
await Order.create_table()

o1 = Order(customer="Acme",   total=299.99)
o2 = Order(customer="Globex", total=149.50)
await o1.save()  # order_number auto-set to 1000
await o2.save()  # order_number auto-set to 1010  (batch=10 pre-allocates)

print(f"Order #{o1.order_number}: {o1.id}")
print(f"Order #{o2.order_number}: {o2.id}")

## 6. Verify the sequence is registered in SurrealDB

In [ ]:
import json
from surrealengine import get_active_connection

active = get_active_connection()
info = await active.client.query("INFO FOR DB")
db_info = info[0] if isinstance(info, list) else info
if isinstance(db_info, dict):
    print("Sequences:", json.dumps(db_info.get("sequences", {}), indent=2, default=str))
else:
    print(info)

## 7. Cleanup

In [ ]:
for inv in await Invoice.objects.all():
    await inv.delete()

for o in await Order.objects.all():
    await o.delete()

print("Cleaned up")